In [32]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [33]:
data = pd.read_csv("../../../data/raw/kathmandu_full_raw_2023_2024.csv")

data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time").reset_index(drop=True)

In [35]:
# Scale pm2_5
scaler = StandardScaler()
pm25_scaled = scaler.fit_transform(data[['pm2_5']])

# Create sliding windows
def create_windows(df, window=12):
    X, y = [], []
    for i in range(len(df) - window):
        X.append(df[i : i+window])
        y.append(df[i+window])
    return np.array(X), np.array(y)

X, y = create_windows(pm25_scaled, window=12)
# X shape: (n-12, 12, 1)
# y shape: (n-12, 1)

# Temporal split
split = int(np.ceil(0.8 * len(X)))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# --- Hyperparameters ---
HIDDEN = 64
LAYERS = 2
LR     = 0.001
EPOCHS = 20
BATCH  = 32

# --- Convert to tensors ---
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)

# --- DataLoader ---
loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                    batch_size=BATCH, shuffle=False)

# --- Model ---
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm   = nn.LSTM(input_size=1, hidden_size=HIDDEN,
                              num_layers=LAYERS, batch_first=True)
        self.linear = nn.Linear(HIDDEN, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.linear(out[:, -1, :])  # take last timestep only

model     = LSTMModel()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# --- Training loop ---
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {loss.item():.4f}")

# --- Predict ---
model.eval()
with torch.no_grad():
    train_preds_scaled = model(X_train_t).numpy()
    test_preds_scaled  = model(X_test_t).numpy()

# --- Inverse transform ---
train_preds = scaler.inverse_transform(train_preds_scaled)
test_preds  = scaler.inverse_transform(test_preds_scaled)
y_train_orig = scaler.inverse_transform(y_train)
y_test_orig  = scaler.inverse_transform(y_test)

# --- Metrics ---
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

train_rmse = mean_squared_error(y_train_orig, train_preds, squared=False)
train_mae  = mean_absolute_error(y_train_orig, train_preds)
train_r2   = r2_score(y_train_orig, train_preds)

test_rmse = mean_squared_error(y_test_orig, test_preds, squared=False)
test_mae  = mean_absolute_error(y_test_orig, test_preds)
test_r2   = r2_score(y_test_orig, test_preds)

print(f"Train RMSE: {train_rmse:.4f} | MAE: {train_mae:.4f} | R2: {train_r2:.4f}")
print(f"Test  RMSE: {test_rmse:.4f} | MAE: {test_mae:.4f} | R2: {test_r2:.4f}")

array([[2.30499308],
       [2.464721  ],
       [2.77387183],
       [2.97997238],
       [2.10404504]])